In [ ]:
# Import Libraries
import pandas as pd
from pybaseball import playerid_lookup
import logging
import re

# Configure Logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def read_csv_file(file_path):
    """
    Reads a CSV file into a pandas DataFrame.
    """
    try:
        df = pd.read_csv(file_path)
        logging.info(f"Successfully read the file: {file_path}")
        return df
    except FileNotFoundError:
        logging.error(f"File not found: {file_path}")
        raise
    except Exception as e:
        logging.error(f"An error occurred while reading the file: {e}")
        raise

def clean_player_column(df):
    """
    Cleans the 'Player' column by removing handedness (RHP/LHP)
    and splits it into 'Last Name' and 'First Name'.
    """
    if 'Player' not in df.columns:
        logging.error("The 'Player' column is missing from the DataFrame.")
        raise KeyError("The 'Player' column is required.")

    # Remove handedness (e.g., 'RHP' or 'LHP') from 'Player' column
    df['Player'] = df['Player'].str.replace(r'\s+\b[RrLl][Hh][Pp]\b', '', regex=True)

    # Split 'Player' into 'Last Name' and 'First Name'
    df[['Last Name', 'First Name']] = df['Player'].str.strip().str.split(',', expand=True)

    # Clean up extra spaces
    df['First Name'] = df['First Name'].str.strip()
    df['Last Name'] = df['Last Name'].str.strip()

    # Handle any cases with suffixes
    def remove_suffix(name):
        suffixes = [' Jr.', ' Sr.', ' II', ' III', ' IV', ' V']
        for suffix in suffixes:
            if name.endswith(suffix):
                name = name[:-len(suffix)]
                break
        return name.strip()

    df['Last Name'] = df['Last Name'].apply(remove_suffix)

    # Insert space after periods in initials in 'First Name' and 'Last Name'
    def insert_space_after_period(name):
        # Example: 'J.P.' becomes 'J. P.'
        name = re.sub(r'\b([A-Za-z])\.', r'\1. ', name)
        # Remove extra spaces if any
        name = re.sub(r'\s+', ' ', name)
        return name.strip()

    df['First Name'] = df['First Name'].apply(insert_space_after_period)
    df['Last Name'] = df['Last Name'].apply(insert_space_after_period)

    logging.info("Cleaned 'Player' column and extracted 'First Name' and 'Last Name'.")
    return df

def get_player_ids(df):
    """
    Looks up player IDs using pybaseball and adds them to the DataFrame.
    Includes checks for availability of player IDs and logs missing entries.
    """
    # Get unique names to minimize API calls
    unique_names = df[['First Name', 'Last Name']].drop_duplicates()
    unique_names['player_id'] = None  # Initialize the player_id column

    missing_ids = []  # To track names without player IDs

    # Iterate over unique names and look up player IDs
    for index, row in unique_names.iterrows():
        first = row['First Name']
        last = row['Last Name']
        try:
            lookup = playerid_lookup(last, first)
            if not lookup.empty:
                # Use the most recent player
                player_info = lookup.sort_values('mlb_played_last', ascending=False).iloc[0]
                player_id = player_info['key_mlbam']
                unique_names.at[index, 'player_id'] = player_id
                logging.debug(f"Found player ID {player_id} for {first} {last}.")
            else:
                # No match found
                logging.warning(f"No player ID found for {first} {last}.")
                missing_ids.append(f"{first} {last}")
        except Exception as e:
            logging.error(f"Error looking up {first} {last}: {e}")
            missing_ids.append(f"{first} {last}")

    # Merge the player IDs back into the original DataFrame
    df = df.merge(unique_names, on=['First Name', 'Last Name'], how='left')

    # Count and display the number of missing player IDs
    missing_count = df['player_id'].isna().sum()
    total_players = len(unique_names)
    logging.info(f"Total unique players: {total_players}")
    logging.info(f"Number of players without player ID: {missing_count}")

    if missing_ids:
        logging.info("Players without IDs:")
        for name in missing_ids:
            logging.info(f"- {name}")

    return df

def main():
    # File path
    savant_file = r"C:\Users\TrevorWhite\Downloads\pitch_movement_noid.csv"
    output_file = r"C:\Users\TrevorWhite\Downloads\pitch_movement_with_ids.csv"

    # Read the savant_data CSV file
    df = read_csv_file(savant_file)

    # Clean the 'Player' column
    df = clean_player_column(df)

    # Get player IDs and merge them into the DataFrame
    df = get_player_ids(df)

    # Save the updated DataFrame to a new CSV file
    df.to_csv(output_file, index=False)
    logging.info(f"Updated DataFrame saved to: {output_file}")

    # Display the first few rows of the updated DataFrame
    print(df.head())

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd

def main():
    # File paths
    savant_data_file = r"C:\Users\TrevorWhite\Downloads\savant_data.csv"
    pitcher_arm_angles_file = r"C:\Users\TrevorWhite\Downloads\pitch_movement_with_ids.csv"
    spin_direction_pitches_file = r"C:\Users\TrevorWhite\Downloads\spin-direction-new.csv"

    # Read the CSV files and ensure player_id columns are loaded as strings
    df_savant = pd.read_csv(savant_data_file, dtype={'player_id': str})
    df_arm_angles = pd.read_csv(pitcher_arm_angles_file, dtype={'player_id': str})
    df_spin_direction = pd.read_csv(spin_direction_pitches_file, dtype={'player_id': str})

    # Rename pitcher column to player_id in df_arm_angles for consistency
    df_arm_angles.rename(columns={'pitcher_id': 'player_id'}, inplace=True)

    # Merge savant data with pitcher arm angles on 'player_id'
    df_combined = pd.merge(df_savant, df_arm_angles, on=['player_id', 'pitch_type', 'year'], how='left')

    # Rename columns for consistency
    #df_combined.rename(columns={'Pitch Type.1': 'PitchType'}, inplace=True)
    df_spin_direction.rename(columns={'api_pitch_type': 'pitch_type'}, inplace=True)
    
    # Now merge on 'player_id' and 'PitchType'
    df_combined = pd.merge(
        df_combined,
        df_spin_direction,
        on=['player_id', 'pitch_type', 'year'],
        how='left'
    )

    # Remove rows with any missing values if needed
  #  df_combined.dropna(inplace=True)

    # Count the remaining rows
   # remaining_rows = len(df_combined)
    #print(f"Remaining rows after dropping NaNs: {remaining_rows}")

    # Ensure player_id is of string type to avoid any issues with decimals
    df_combined['player_id'] = df_combined['player_id'].astype(str)
    
    # Display all columns by default in the console
    pd.set_option('display.max_columns', None)

    # Save the DataFrame to the environment by assigning it to a global variable if needed
    global combined_df
    combined_df = df_combined

if __name__ == "__main__":
    main()


In [ ]:
print(len(combined_df))
combined_df.dropna(inplace=True)
remaining_rows = len(combined_df)
print(f"Remaining rows after dropping NaNs: {remaining_rows}")

In [ ]:
import requests
import pandas as pd
import time
from typing import List

def parse_height(height_str: str) -> float:
    """
    Parses the height string in the format 'ft in' and converts it to feet units.
    
    Parameters:
    height_str: Height string in the format 'ft in'

    Returns:
    Height in ft (float)
    """
    feet, inches = height_str.split("' ")
    feet = int(feet)
    inches = int(inches.replace('"', ''))
    height_in_feet = feet + inches / 12.0
    return height_in_feet

def chunk_list(lst: List[int], chunk_size: int) -> List[List[int]]:
    """
    Splits a list into chunks of specified size.
    
    Parameters:
    lst: List to be split.
    chunk_size: Size of each chunk.
    
    Returns:
    List of chunks.
    """
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

def get_player_heights(player_ids: List[int]) -> pd.DataFrame:
    """
    Fetches the heights and weights of players given their IDs.
    
    Parameters:
    player_ids: List of player IDs.
    
    Returns:
    pandas DataFrame of player IDs, heights, and weights.
    """
    all_data = []

    for chunk in chunk_list(player_ids, 500):
        player_id_str = ','.join(map(str, chunk))
        url = f'https://statsapi.mlb.com/api/v1/people?personIds={player_id_str}&fields=people,id,height,weight'

        response = requests.get(url)
        response.raise_for_status()  # Raise an error if the request fails

        json_data = response.json()['people']
        data = pd.DataFrame(json_data)

        # Convert height to feet
        data['height'] = data['height'].apply(parse_height)
        all_data.append(data)

        # Avoid overloading the server
        time.sleep(0.5)

    # Concatenate all chunks into a single DataFrame
    result_df = pd.concat(all_data, ignore_index=True)
    return result_df[['id', 'height', 'weight']]

# Extract unique player IDs from df_comb_cleaned
unique_player_ids = combined_df['player_id'].unique()

# Fetch player heights and weights
player_data = get_player_heights(unique_player_ids)

# Rename 'id' column to 'player_id' for merging
player_data = player_data.rename(columns={'id': 'player_id'})

# Convert player_id in both DataFrames to the same type
combined_df['player_id'] = combined_df['player_id'].astype(int)
player_data['player_id'] = player_data['player_id'].astype(int)


combined_df = combined_df.merge(player_data, on='player_id', how='left')

# Display the updated DataFrame
print(combined_df.head())

In [ ]:
pd.set_option('display.max_columns', None)

pd.set_option('display.max_rows', None)


In [ ]:
combined_df.head(2000)

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

def signed_minutes_from_12(row):
    time_str = row['hawkeye_measured_clock_label']
    pitch_hand = row['pitch_hand_y']
    # Check for missing values
    if pd.isnull(time_str) or pd.isnull(pitch_hand):
        return np.nan
    # Convert to string and strip whitespace
    time_str = str(time_str).strip()
    pitch_hand = str(pitch_hand).strip()
    if time_str == '':
        return np.nan
    try:
        # Parse the time string into a datetime object
        time_obj = datetime.strptime(time_str, '%I:%M')
        # Convert time to minutes since 12:00
        total_minutes = (time_obj.hour % 12) * 60 + time_obj.minute  # 0 to 719
        # Calculate signed difference based on pitcher's hand
        if total_minutes == 0:
            difference = 0
        elif total_minutes < 360:
            # Times after 12:00 up to 5:59
            if pitch_hand == 'R':
                difference = total_minutes  # Positive minutes
            else:
                difference = -total_minutes  # Negative minutes
        else:
            # Times from 6:00 up to 11:59
            minutes_to_12 = 720 - total_minutes
            if pitch_hand == 'R':
                difference = -minutes_to_12  # Negative minutes
            else:
                difference = minutes_to_12  # Positive minutes
        return difference
    except ValueError:
        # If parsing fails, return NaN
        return np.nan

# Apply the function to the DataFrame row-wise
combined_df['minutes_past_12'] = combined_df.apply(signed_minutes_from_12, axis=1)

# Display the updated DataFrame
print(combined_df[['player_id', 'hawkeye_measured_clock_label', 'pitch_hand_y', 'minutes_past_12']])


In [ ]:
import pandas as pd
import numpy as np
import re

# Assuming 'hb' is the column in your DataFrame 'df'
def clean_hb(value):
    if pd.isnull(value):
        return np.nan
    value_str = str(value).strip()
    # Use regular expression to extract number and suffix
    match = re.match(r'^([-\d\.]+)(GLV|ARM)$', value_str)
    if match:
        number_part = float(match.group(1))
        suffix = match.group(2)
        if suffix == 'GLV':
            return -abs(number_part)
        elif suffix == 'ARM':
            return abs(number_part)
    else:
        # Handle values without expected suffixes
        try:
            number_part = float(re.findall(r'[-\d\.]+', value_str)[0])
            return number_part
        except (IndexError, ValueError):
            return np.nan

# Apply the function to the 'hb' column
combined_df['hb_cleaned'] = combined_df['hb'].apply(clean_hb)

# Display the original and cleaned 'hb' columns
print(combined_df[['hb', 'hb_cleaned']].head())


In [ ]:
print(combined_df[['hb', 'hb_cleaned']].head(1000))

In [ ]:
# Take the absolute value of 'xRel' in df_combined


combined_df = combined_df[combined_df['arm_angle'] > -20]

combined_df['xRel'] = np.where(
    combined_df['pitch_hand_y'] == 'R',
    combined_df['release_pos_x'] * -1,
    combined_df['release_pos_x']
)

combined_df['plate_x'] = np.where(
    combined_df['pitch_hand_y'] == 'L',
    combined_df['plate_x'] * -1,
    combined_df['plate_x']
)



# Create boolean columns for each unique value in the PitchType column
combined_df = pd.get_dummies(combined_df, columns=['pitch_type'], prefix='', prefix_sep='')

# Step 1: Remove rows where player_id is 643511
combined_df = combined_df[combined_df['player_id'] != '643511']


# Specify the columns to drop
columns_to_drop = [
    'Player', '_1', 'Last Name', 'First Name', 'pitcher_name', 'year', 
    'pitch_hand_x', 'n_pitches', 'team_id', 'pitch_hand_y', 'last_name, first_name_y', 'api_pitch_type',
     'hawkeye_measured_clock_label', 'Column1', 'release_pos_x'
]



# Drop the specified columns
df_comb_cleaned = combined_df.drop(columns=columns_to_drop, errors='ignore')

In [ ]:
sorted_df = df_comb_cleaned.sort_values(by='hb_cleaned')

# Display the first 1000 rows of the sorted DataFrame
sorted_df.head(1000)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
import numpy as np

# Initial and extended feature sets
initial_features = ['xRel', 'zRel', 'FF', 'Ext', 'height']
extended_features = initial_features + ['ivb', 'Arm Side Movement', 'spin', 'tilt_min']

# All features including target
all_features = list(set(initial_features + extended_features + ['ball_angle']))

print("All features:", all_features)

# Ensure the data is numeric and handle any NaN values
print("\nConverting all_features to numeric...")
df_comb_cleaned[all_features] = df_comb_cleaned[all_features].apply(pd.to_numeric, errors='coerce')

# Drop rows with any NaN values in the features or target
df_comb_cleaned.dropna(subset=all_features, inplace=True)

# Split data into features and target
X_initial = df_comb_cleaned[initial_features]
X_extended = df_comb_cleaned[extended_features]
y = df_comb_cleaned['ball_angle']

# Add constant term
X_initial = sm.add_constant(X_initial)
X_extended = sm.add_constant(X_extended)

# Split into training and testing sets
X_train_initial, X_test_initial, y_train_initial, y_test = train_test_split(
    X_initial, y, test_size=0.2, random_state=42
)
X_train_extended, X_test_extended, y_train_extended, _ = train_test_split(
    X_extended, y, test_size=0.2, random_state=42
)

# Forcefully convert to numeric arrays and replace non-numeric entries with NaN
X_train_initial_array = np.array(X_train_initial, dtype=np.float64)
y_train_initial_array = np.array(y_train_initial, dtype=np.float64)
X_test_initial_array = np.array(X_test_initial, dtype=np.float64)
y_test_array = np.array(y_test, dtype=np.float64)

X_train_extended_array = np.array(X_train_extended, dtype=np.float64)
y_train_extended_array = np.array(y_train_extended, dtype=np.float64)
X_test_extended_array = np.array(X_test_extended, dtype=np.float64)

# Check for NaNs and Infs in the converted arrays
print("\nAny NaNs in X_train_initial_array?", np.isnan(X_train_initial_array).any())
print("Any NaNs in y_train_initial_array?", np.isnan(y_train_initial_array).any())
print("Any Infs in X_train_initial_array?", np.isinf(X_train_initial_array).any())
print("Any Infs in y_train_initial_array?", np.isinf(y_train_initial_array).any())

# Drop rows with NaN or Inf in X_train_initial_array and align y_train accordingly
valid_indices = ~np.isnan(X_train_initial_array).any(axis=1)
X_train_initial_array = X_train_initial_array[valid_indices]
y_train_initial_array = y_train_initial_array[valid_indices]

# Similarly handle NaNs/Infs for extended features
valid_indices_ext = ~np.isnan(X_train_extended_array).any(axis=1)
X_train_extended_array = X_train_extended_array[valid_indices_ext]
y_train_extended_array = y_train_extended_array[valid_indices_ext]

# Fit the initial model
print("\nFitting the initial model...")
try:
    model_initial = sm.OLS(y_train_initial_array, X_train_initial_array).fit()
    y_pred_initial = model_initial.predict(X_test_initial_array)
    rmse_initial = np.sqrt(mean_squared_error(y_test_array, y_pred_initial))

    print("\nInitial Model Summary:")
    print(model_initial.summary())
    print(f"Initial Model RMSE: {rmse_initial:.2f}")
except Exception as e:
    print("Error fitting the initial model:", e)

# Fit the extended model
print("\nFitting the extended model...")
try:
    model_extended = sm.OLS(y_train_extended_array, X_train_extended_array).fit()
    y_pred_extended = model_extended.predict(X_test_extended_array)
    rmse_extended = np.sqrt(mean_squared_error(y_test_array, y_pred_extended))

    print("\nExtended Model Summary:")
    print(model_extended.summary())
    print(f"Extended Model RMSE: {rmse_extended:.2f}")
except Exception as e:
    print("Error fitting the extended model:", e)


In [ ]:
print("Data types of X_train_initial:")
print(X_train_initial.dtypes)
print("\nData types of y_train:")
print(y_train.dtypes)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt

# Initial and extended feature sets
initial_features = ['xRel', 'zRel', 'FF', 'Ext', 'height']
extended_features = initial_features + ['ivb', 'Arm Side Movement', 'spin', 'tilt_min']

# All features including target
all_features = extended_features + ['ball_angle']

# Ensure the data is numeric
df_comb_cleaned[all_features] = df_comb_cleaned[all_features].apply(pd.to_numeric, errors='coerce')

# Drop rows with any NaN values
df_comb_cleaned.dropna(subset=all_features, inplace=True)

# Split data into features and target
X = df_comb_cleaned[extended_features]
y = df_comb_cleaned['ball_angle']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Initialize the XGBRegressor
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Gradient Boosting Regression RMSE: {rmse:.2f}")

# Plot feature importance
xgb.plot_importance(model)
plt.show()


In [ ]:
df_comb_cleaned.head(1000)


#negative plate x = toward RHB

In [ ]:


import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt

# Assuming df_comb_cleaned is your cleaned DataFrame
df = df_comb_cleaned.copy()

# Initial and extended feature sets
initial_features = ['xRel', 'release_pos_z', 'FF', 'release_extension', 'height', 'velocity', 'minutes_past_12']
existing_extended_features = ['ivb', 'hb_cleaned', 'spin_rate', 'plate_x', 'plate_z']
interaction_features = [
    'height_Ext',
    'height_zRel',
    'height_xRel',
    'tilt_min_xRel',
    'tilt_min_zRel',
    'ivb_zRel',
    'ArmSideMovement_xRel',
    'spin_ivb',
    'spin_ArmSideMovement',
    'Ext_zRel',
    'Ext_xRel',
    'ff_hb',
    'ff_ivb',
    'ivb_plate_z',
    'ArmSideMovement_plate_x',
    'FF_plate_z',
    'FF_plate_x',
    'spin_plate_z',
    'spin_plate_x'
    
]

# Compute the new interaction features
df['height_Ext'] = df['height'] * df['release_extension']
df['Ext_zRel'] = df['release_pos_z'] * df['release_extension']
df['Ext_xRel'] = df['xRel'] * df['release_extension']
df['height_zRel'] = df['height'] * df['release_pos_z']
df['height_xRel'] = df['height'] * df['xRel']

df['plate_x_plate_z'] = df['plate_x'] * df['plate_z']
df['ivb_plate_z'] = df['ivb'] * df['plate_z']
df['ArmSideMovement_plate_x'] = df['hb_cleaned'] * df['plate_x']
df['FF_plate_z'] = df['FF'] * df['plate_z']
df['FF_plate_x'] = df['FF'] * df['plate_x']
df['spin_plate_z'] = df['spin_rate'] * df['plate_z']
df['spin_plate_x'] = df['spin_rate'] * df['plate_x']

df['tilt_min_xRel'] = df['minutes_past_12'] * df['xRel']
df['tilt_min_zRel'] = df['minutes_past_12'] * df['release_pos_z']
df['ivb_zRel'] = df['ivb'] * df['release_pos_z']
df['ArmSideMovement_xRel'] = df['hb_cleaned'] * df['xRel']
df['spin_ivb'] = df['spin_rate'] * df['ivb']
df['spin_ArmSideMovement'] = df['spin_rate'] * df['hb_cleaned']
df['ff_hb'] = df['FF'] * df['hb_cleaned']
df['ff_ivb'] = df['FF'] * df['ivb']

# Extended features include initial, existing extended, and new interaction features
extended_features = initial_features + existing_extended_features + interaction_features

# All features including target
all_features = extended_features + ['arm_angle']

# Ensure the data is numeric
df[all_features] = df[all_features].apply(pd.to_numeric, errors='coerce')

# Drop rows with any NaN or Inf values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(subset=all_features, inplace=True)

# Split data into features and target
X = df[extended_features]
y = df['arm_angle']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define a less aggressive parameter grid for hyperparameter tuning
# Define a moderately extensive parameter grid for hyperparameter tuning
param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9],
    'gamma': [0, 0.1],
    'min_child_weight': [1, 3],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [1, 10]
}



# Initialize the XGBRegressor
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42
)

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    verbose=1,
    n_jobs=-1
)

# Perform the grid search on the training data
grid_search.fit(X_train, y_train)

# Best parameters
print("Best parameters found: ", grid_search.best_params_)

# Use the best estimator
best_model = grid_search.best_estimator_

# Make predictions with the best model
y_pred_best = best_model.predict(X_test)

# Calculate RMSE
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
print(f"Optimized Gradient Boosting Regression RMSE: {rmse_best:.2f}")

# Plot feature importance
xgb.plot_importance(best_model, max_num_features=20)
plt.show()


In [ ]:
import joblib

# Save the model to a file
joblib.dump(best_model, 'best_xgboost_model.pkl')
print("Model saved successfully!")


In [ ]:
# Save the model to a JSON file
best_model.save_model('best_xgboost_model.json')
print("Model saved successfully in JSON format!")


In [ ]:
from scipy.stats import norm

# Calculate residuals
residuals = y_test - y_pred_best

# Standard deviation of residuals
sigma = np.std(residuals)

# Compute confidence intervals for predictions
confidence_level = 0.90
z = norm.ppf((1 + confidence_level) / 2)
lower_bound = y_pred_best - z * sigma
upper_bound = y_pred_best + z * sigma

print(f"95% Confidence Interval: Mean Prediction ± {z * sigma:.2f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming y_test and y_pred_best are your actual and predicted values
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_best, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs. Predicted Values')
plt.show()
